# Volume 7, Chapter 3: Operators, Spectra and Banach Algebras (Python companion)

Exercises first, solutions at the end. Spectral theory on a computer: iterative linear solvers and the spectral radius, pseudospectra and transient growth, integral equations by the Nyström method, and Wiener's lemma in numbers. Requirements: numpy and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P3.1 (★★) CS application: iterative solvers and the spectral radius.** Discretise $-u''=1$ on $[0,1]$ with $u(0)=u(1)=0$ by finite differences with $n$ interior points, giving $Au=b$ with $A$ tridiagonal. Solve it by the Jacobi iteration $u_{k+1}=D^{-1}(b-(A-D)u_k)$ and by Gauss–Seidel. Measure the error after $k$ steps and compare its decay rate with the spectral radius of the iteration matrix (Exercise 3.5 and Gelfand's formula, Theorem 3.6). How does the number of steps needed scale with $n$?

**P3.2 (★) Pseudospectra: when eigenvalues lie.** Let $J$ be the $N\times N$ Jordan block of Figure 3.1. (a) Perturb its bottom-left corner by $\varepsilon=10^{-10}$ and compute the eigenvalues: where are they, and why? (b) Take $M=0.9I+J$, whose spectral radius is $0.9$: plot $\|M^k\|$ for $k\le1000$ and $N=10$. Gelfand's formula says $\|M^k\|^{1/k}\to0.9$; what happens on the way?

**P3.3 (★★) Integral equations on a computer.** Solve the Fredholm equation $f(x)-\lambda\int_0^1G(x,y)f(y)\,dy=1$ with $G(x,y)=\min(x,y)-xy$ by the Nyström method (replace the integral by a quadrature rule and solve the linear system) for $\lambda=5$, $\lambda=9$ and $\lambda=\pi^2$, and compare with the exact solution. Then solve the Volterra equation $f(x)=1+\int_0^xf$ by Picard iteration and check the factorial convergence of Example 3.7.

**P3.4 (★) Wiener's lemma in numbers.** For $f(t)=1+0.6e^{it}+0.3e^{-3it}$ (Figure 3.2), compute the Fourier coefficients of $\frac1f$ with the FFT and check that they are absolutely summable. How fast do they decay? Repeat with $f_a(t)=1+ae^{it}$ for $a=0.5,\,0.9,\,0.99,\,1$, and relate $\|\frac1{f_a}\|_W$ to the distance from the curve to $0$.

## Solutions

### P3.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def poisson(n):
    h = 1 / (n + 1)
    A = (np.diag(2 * np.ones(n)) - np.diag(np.ones(n - 1), 1) - np.diag(np.ones(n - 1), -1)) / h ** 2
    x = np.linspace(h, 1 - h, n)
    return A, np.ones(n), x * (1 - x) / 2          # exact solution of -u''=1 (and of the scheme)

def iterate(A, b, exact, method, steps):
    D = np.diag(A); u = np.zeros_like(b); errs = []
    for _ in range(steps):
        if method == "jacobi":
            u = (b - (A @ u - D * u)) / D
        else:                                       # Gauss-Seidel: use new values at once
            for i in range(len(b)):
                u[i] = (b[i] - A[i] @ u + D[i] * u[i]) / D[i]
        errs.append(np.max(np.abs(u - exact)))
    return np.array(errs)

for n in [10, 20, 40]:
    A, b, exact = poisson(n)
    D = np.diag(np.diag(A)); L = np.tril(A, -1); U = np.triu(A, 1)
    rJ = max(abs(np.linalg.eigvals(-np.linalg.solve(D, L + U))))
    rG = max(abs(np.linalg.eigvals(-np.linalg.solve(D + L, U))))
    eJ = iterate(A, b, exact, "jacobi", 2000); eG = iterate(A, b, exact, "gs", 1000)
    kJ = np.nonzero(eJ > 1e-11)[0][-1]; kG = np.nonzero(eG > 1e-11)[0][-1]   # stop before the rounding floor
    obsJ = (eJ[kJ] / eJ[kJ // 2]) ** (1 / (kJ - kJ // 2)); obsG = (eG[kG] / eG[kG // 2]) ** (1 / (kG - kG // 2))
    print(f"n={n:3d}: r(Jacobi)={rJ:.5f} observed {obsJ:.5f} | r(GS)={rG:.5f} observed {obsG:.5f} | cos(pi h)={np.cos(np.pi / (n + 1)):.5f}")
    print(f"        steps to gain a factor 10: Jacobi {np.log(10) / -np.log(rJ):7.1f}, Gauss-Seidel {np.log(10) / -np.log(rG):7.1f}")
A, b, exact = poisson(20)
plt.semilogy(iterate(A, b, exact, "jacobi", 1500), label='Jacobi')
plt.semilogy(iterate(A, b, exact, "gs", 1500), label='Gauss-Seidel')
plt.xlabel('step'); plt.ylabel('max error'); plt.legend(); plt.title('n = 20'); plt.show()

The error decays geometrically at exactly the rate $r$ of the iteration matrix: the observed ratios match the spectral radii to five digits. That is Gelfand's formula at work, $\|M^k\|^{1/k}\to r(M)$, since the error after $k$ steps is $M^k$ applied to the initial error. For this matrix $r(\text{Jacobi})=\cos(\pi h)$ with $h=\frac1{n+1}$ (the eigenvectors are discrete sines), and Gauss–Seidel squares it, so it is twice as fast. Both are slow: $1-\cos\pi h\approx\frac{\pi^2h^2}2$, so the number of steps grows like $n^2$, quadrupling each time $n$ doubles. The fine grid sees the smooth error modes as almost invariant. Curing this is the idea of multigrid (smooth on the fine grid, correct on a coarse one) and of Krylov methods such as conjugate gradients, whose rate depends on $\sqrt{\text{condition number}}$, so on $n$ rather than $n^2$.

### P3.2

In [ ]:
N = 64
J = np.diag(np.ones(N - 1), 1)
Je = J.copy(); Je[-1, 0] = 1e-10
ev = np.linalg.eigvals(Je)
print("moduli of the eigenvalues after a 1e-10 perturbation: min %.4f, max %.4f" % (abs(ev).min(), abs(ev).max()))
print("1e-10 ** (1/64) =", 1e-10 ** (1 / N))
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].plot(ev.real, ev.imag, '.'); ax[0].add_patch(plt.Circle((0, 0), 1, fill=False, ls='--'))
ax[0].set_aspect('equal'); ax[0].set_title('eigenvalues of J + 1e-10 E_{N1}')
N = 10
M = 0.9 * np.eye(N) + np.diag(np.ones(N - 1), 1)
norms = []; P = np.eye(N)
for k in range(1001):
    norms.append(np.linalg.norm(P, 2)); P = P @ M
norms = np.array(norms)
ax[1].semilogy(norms); ax[1].semilogy(0.9 ** np.arange(1001), '--', label='0.9^k')
ax[1].legend(); ax[1].set_title('||M^k||, M = 0.9 I + J, N = 10'); plt.show()
k = np.argmax(norms)
print(f"max ||M^k|| = {norms[k]:.3e} at k = {k}")
for k in [100, 300, 1000]:
    print(f"||M^{k}||^(1/{k}) = {norms[k] ** (1 / k):.4f}")

(a) The $64$ eigenvalues jump from $0$ to a circle of radius $10^{-10/64}\approx0.70$: the characteristic polynomial of the perturbed matrix is $z^{64}-10^{-10}$. A perturbation at the level of rounding errors moves the spectrum by $0.7$, which is why the computed eigenvalues of non-normal matrices can be meaningless, and why Figure 3.1 draws the pseudospectra instead: the set where $\|(z-J)^{-1}\|>\frac1\varepsilon$ is exactly the set of eigenvalues of all perturbations of norm less than $\varepsilon$. (b) The spectral radius $0.9$ promises decay, and eventually delivers it, but first $\|M^k\|$ climbs to about $10^{8}$ (near $k=90$) before turning around, and it falls below $1$ only after several hundred steps. The roots $\|M^k\|^{1/k}$ do tend to $0.9$, but slowly: about $1.2$ at $k=100$, $1.06$ at $k=300$, $0.95$ at $k=1000$, because $M^k=\sum_j\binom kj0.9^{k-j}J^j$ carries the polynomial factor $\binom k9$. Gelfand's formula is a statement about the limit only. In a dynamical system, an iterative solver or a neural network's recurrent layer, this transient growth is what you actually see, and it is governed by the pseudospectra, not the spectrum (Trefethen and Embree, *Spectra and Pseudospectra*).

### P3.3

In [ ]:
from numpy.polynomial.legendre import leggauss
def nystrom(lam, n):
    t, w = leggauss(n); x = (t + 1) / 2; w = w / 2
    G = np.minimum.outer(x, x) - np.outer(x, x)
    f = np.linalg.solve(np.eye(n) - lam * G * w, np.ones(n))
    return x, f, np.linalg.cond(np.eye(n) - lam * G * w)

def exact(lam, x):          # f'' + lam f = 0 with f(0)=f(1)=1, for 0 < lam
    s = np.sqrt(lam)
    return np.cos(s * x) + (1 - np.cos(s)) / np.sin(s) * np.sin(s * x)

for lam in [5.0, 9.0, np.pi ** 2]:
    for n in [5, 10, 20]:
        x, f, c = nystrom(lam, n)
        err = np.max(np.abs(f - exact(lam, x))) if abs(lam - np.pi ** 2) > 1e-9 else float('nan')
        print(f"lambda={lam:8.5f}, n={n:2d}: max error {err:.2e}, condition number {c:.2e}")
# Volterra: f = 1 + V f by Picard iteration on a grid
x = np.linspace(0, 1, 2001)
def V(f):
    return np.concatenate([[0], np.cumsum((f[1:] + f[:-1]) / 2 * np.diff(x))])
f = np.ones_like(x)
for k in range(1, 13):
    f = 1 + V(f)
    print(f"Picard step {k:2d}: max |f - e^x| = {np.max(np.abs(f - np.exp(x))):.2e}", end="   ")
    print(f"e/(k+1)! = {np.e / np.prod(np.arange(1, k + 2)):.2e}")

The kernel $G$ is the Green's function of $-u''$ with zero boundary values, so $f-\lambda Gf=1$ means $f''+\lambda f=0$, $f(0)=f(1)=1$, solved in closed form above. For $\lambda=5$, Gauss–Legendre Nyström has errors of $0.17$, $0.04$, $0.01$ with $5$, $10$, $20$ nodes: they shrink like $n^{-2}$, not spectrally, because $G$ has a corner on the diagonal, which caps the accuracy of any quadrature rule applied to $y\mapsto G(x,y)f(y)$. For $\lambda=9$ the errors are much larger: the solution contains the factor $\frac{1-\cos\sqrt\lambda}{\sin\sqrt\lambda}$, which is about $14$ at $\lambda=9$, so the equation amplifies every quadrature error by a factor of that order, even though the discrete systems are still well conditioned. At $\lambda=\pi^2$, the Fredholm alternative (Theorem 3.4) bites: $\frac1{\pi^2}$ is an eigenvalue of $G$ (eigenfunction $\sin\pi x$), the homogeneous equation has a solution, and the inhomogeneous one is solvable only if $1\perp\sin\pi x$, which fails. The discrete system is still invertible, but its condition number grows without bound with $n$ (about $16$, $62$, $239$: roughly like $n^2$), the numerical shadow of non-existence; there is no solution to compare with, hence the `nan`. For the Volterra equation, the Picard iterates are the partial sums $\sum_{j\le k}\frac{x^j}{j!}$ of $e^x$, and the error after $k$ steps stays below $\frac{e}{(k+1)!}$, as the bound $\|V^n\|\le\frac1{(n-1)!}$ of Example 3.7 predicts, until it hits the floor of about $5\times10^{-8}$ set by the trapezoid rule on the grid. A spectral radius of $0$ means the Neumann series converges faster than any geometric series.

### P3.4

In [ ]:
M = 4096
t = 2 * np.pi * np.arange(M) / M
def coeffs(f):
    return np.fft.fft(1 / f(t)) / M            # c_n for n = 0..M/2 and negative n at the end
f = lambda t: 1 + 0.6 * np.exp(1j * t) + 0.3 * np.exp(-3j * t)
c = coeffs(f); n = np.fft.fftfreq(M, 1 / M).astype(int)
print("min |f| on the circle:", np.abs(f(t)).min())
print("||1/f||_W = sum |c_n| =", np.abs(c).sum())
for k in [1, 10, 20, 40, 80]:
    print(f"|c_{k:3d}| = {abs(c[n == k][0]):.2e}   |c_{-k:4d}| = {abs(c[n == -k][0]):.2e}")
for a in [0.5, 0.9, 0.99, 1.0]:
    fa = lambda t: 1 + a * np.exp(1j * t)
    with np.errstate(divide='ignore', invalid='ignore'):
        ca = coeffs(fa)
    print(f"a={a}: distance to 0 = {1 - a:.2f}, sum |c_n| = {np.abs(ca).sum():.4g}, 1/(1-a) = {1 / (1 - a) if a < 1 else np.inf:.4g}")
plt.semilogy(np.arange(1, 120), np.abs(c[1:120]), label='n > 0'); plt.semilogy(np.arange(1, 120), np.abs(c[-1:-120:-1]), label='n < 0')
plt.xlabel('|n|'); plt.ylabel('|c_n| of 1/f'); plt.legend(); plt.show()

The Fourier coefficients of $\frac1f$ decay geometrically, and their sum of moduli is finite, as Wiener's lemma promises. It is exactly $10=\frac1{\min|f|}$, and that is no accident: $f(t+\pi)=1-u(t)$ with $u=0.6e^{it}+0.3e^{-3it}$, and $\frac1{1-u}=\sum_ku^k$ has nonnegative Fourier coefficients, whose sum is its value at $t=0$, namely $\frac1{1-0.9}$; shifting $t$ by $\pi$ only changes the signs of the coefficients.  The proof of the lemma gives no rate at all: geometric decay here comes from extra information, namely that $f$ is a trigonometric polynomial, so $\frac1f$ extends holomorphically to an annulus around the circle (a Chapter 9 story). For $f_a=1+ae^{it}$ one can compute by hand: $\frac1{f_a}=\sum_{n\ge0}(-a)^ne^{int}$, so $\|\frac1{f_a}\|_W=\frac1{1-a}$, exactly the printed values, and the reciprocal of the distance from the curve to $0$. As $a\to1$ the curve touches $0$ and the norm blows up; at $a=1$, $f_1$ vanishes at $t=\pi$, $\frac1{f_1}$ is not even bounded, and the FFT returns `inf` or `nan`. The inverse in a Banach algebra is continuous but not uniformly so: $\|a^{-1}\|$ can be huge when $a$ is close to the non-invertible elements. This is the infinite-dimensional version of the condition number in P3.1.